# Lab 02: How Spark plans a query

**Time:** about 2 h 30 min.

**Checklist questions this lab gives you evidence for:**<br><span style="color:#d93025"><b>1.6 Catalyst Optimizer</b></span><br><span style="color:#d93025"><b>1.8 Tungsten</b></span><br><span style="color:#d93025"><b>1.13 DataFrame vs. RDD</b></span>

## Data information
NYC Yellow Taxi trips, January 2024 ([source](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page), [data dictionary](https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf)): 2,964,624 trips. The setup cell downloads it. Real data, so some rows are dirty (dates outside January, negative fares).

**trips** (`data/yellow_tripdata_2024-01.parquet`):
* *tpep_pickup_datetime*, *tpep_dropoff_datetime*: when the meter started and stopped
* *PULocationID*, *DOLocationID*: pickup and dropoff zone (joins to *zones.LocationID*)
* *trip_distance*: miles; *passenger_count*: number of passengers
* *payment_type*: 1 = credit card, 2 = cash, 3 = no charge, 4 = dispute, 0 = unknown
* *fare_amount*, *tip_amount*, *total_amount*: dollars
* *VendorID*: the company that sent the record

**zones** (`data/taxi_zone_lookup.csv`, 265 rows): *LocationID*, *Borough*, *Zone*, *service_zone*


## How to work
* Write your answer in the code cell under each question, then run the check cell below it: ✅ correct, ❌ try again.
* Lab 01 gives code skeletons with `___` blanks for the first questions only. From then on, and in every lab after it, you write the code yourself; cells only contain the objects a question gives you.
* When a question says *guess first*, write your guess as a comment before running.
* Keep the Spark UI open (the setup cell prints its address).
* Stuck for more than 10 minutes? Read the linked docs page, then the reference solution.


## Documentation
Everything you need is on these pages. The questions do not repeat them: read, try things out in a cell, and find the answers yourself.

* [DataFrame.groupBy](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.groupBy.html)
* [SELECT reference](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select.html)
* [GroupedData.agg](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.GroupedData.agg.html)
* [DataFrame.join](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.join.html)
* [EXPLAIN](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-explain.html)
* [Deep Dive into Spark SQL's Catalyst Optimizer](https://www.databricks.com/blog/2015/04/13/deep-dive-into-spark-sqls-catalyst-optimizer.html)
* [Spark SQL configuration](https://spark.apache.org/docs/latest/configuration.html#runtime-sql-configuration)
* [functions.udf](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.udf.html)
* [Parquet filter pushdown](https://spark.apache.org/docs/latest/sql-data-sources-parquet.html#configuration)
* [Parquet files](https://spark.apache.org/docs/latest/sql-data-sources-parquet.html)
* [Spark UI: SQL tab](https://spark.apache.org/docs/latest/web-ui.html#sql-tab)
* [Project Tungsten](https://www.databricks.com/blog/2015/04/28/project-tungsten-bringing-spark-closer-to-bare-metal.html)
* [RDD.reduceByKey](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.RDD.reduceByKey.html)
* [functions.hour](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.hour.html)
* [ANALYZE TABLE](https://spark.apache.org/docs/latest/sql-ref-syntax-aux-analyze-table.html)

## Setup

In [ ]:
# Setup: run once. Downloads the data, starts Spark, defines the helpers the tasks use.
import hashlib, json, os, re, sys, time, urllib.request, contextlib, io
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)   # executors use this kernel's Python
from pyspark.sql import SparkSession, functions as F

os.makedirs("data", exist_ok=True)
for name, url in {
    "yellow_tripdata_2024-01.parquet": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet",
    "taxi_zone_lookup.csv": "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv",
}.items():
    if not os.path.exists(f"data/{name}"):
        print("downloading", name); urllib.request.urlretrieve(url, f"data/{name}")

def new_session(master="local[4]"):
    s = (SparkSession.builder.master(master).appName("day02_lab")
         .config("spark.ui.showConsoleProgress", "false")
         # Spark 4.2.0 bug (SPARK-60294): Python UDFs on Parquet can hang without this.
         .config("spark.sql.execution.arrow.pythonUDF.columnarInput.enabled", "false")
         .getOrCreate())
    s.sparkContext.setLogLevel("ERROR")
    s.conf.set("spark.sql.adaptive.enabled", "false")   # AQE comes in Lab 03; off = exact job/stage counts
    return s

spark = new_session()
print(f"Spark {spark.version} | UI: {spark.sparkContext.uiWebUrl}")

# --- helpers --------------------------------------------------------------------
def _tracker():
    return spark.sparkContext.statusTracker()

def _run_group(action):
    sc = spark.sparkContext
    group = f"g{time.time_ns()}"
    sc.setJobGroup(group, group)
    try:
        action()
    finally:
        sc.setLocalProperty("spark.jobGroup.id", None)
    return [_tracker().getJobInfo(j) for j in sorted(_tracker().getJobIdsForGroup(group))]

def jobs_run(action):
    """How many Spark jobs calling action() starts."""
    return len(_run_group(action))

def stages_run(action):
    """How many stages those jobs had in total."""
    return sum(len(j.stageIds) for j in _run_group(action))

def last_stage_tasks(action):
    """Number of tasks in the last stage action() ran."""
    jobs = _run_group(action)
    last = max(s for j in jobs for s in j.stageIds)
    return _tracker().getStageInfo(last).numTasks

def failed_attempts():
    """Failed task attempts in the most recent failed stage (from the Spark UI's REST API)."""
    sc = spark.sparkContext
    base = f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}"
    failed = json.load(urllib.request.urlopen(base + "/stages?status=failed"))
    if not failed:
        return 0
    st = max(failed, key=lambda s: s["stageId"])
    tasks = json.load(urllib.request.urlopen(f"{base}/stages/{st['stageId']}/{st['attemptId']}/taskList?length=500"))
    return sum(1 for t in tasks if t["status"] == "FAILED")

def plan(df, mode="formatted"):
    """explain() output as a string."""
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        df.explain(mode)
    return buf.getvalue()

def optimized(df):
    """The optimized logical plan as a string."""
    return df._jdf.queryExecution().optimizedPlan().toString()

def operators(df):
    """Operator names of the physical plan tree, top to bottom."""
    tree = plan(df).split("\n\n")[0].splitlines()[1:]
    return [re.sub(r"\s*\(\d+\)$", "", line).strip(" +-*:") for line in tree]

def rules_fired(df):
    """Names of the Catalyst rules that changed this query's plan."""
    qe = df._jdf.queryExecution(); qe.executedPlan()
    it, fired = qe.tracker().rules().iterator(), set()
    while it.hasNext():
        kv = it.next()
        if kv._2().numEffectiveInvocations() > 0:
            fired.add(kv._1().split(".")[-1].split("$")[-1])
    return fired

def estimate(df):
    """(sizeInBytes, rowCount or None) the optimizer estimates for df."""
    stats = df._jdf.queryExecution().optimizedPlan().stats()
    rows = int(str(stats.rowCount().get())) if stats.rowCount().isDefined() else None
    return int(str(stats.sizeInBytes())), rows

def timeit(action):
    start = time.perf_counter(); action(); return round(time.perf_counter() - start, 2)

# --- checks: compare a hash of your answer with the expected one ------------------
_EXPECTED = {
 "2.1": "c324630dfd340c8d",
 "2.2": "bd761c9fd162d887",
 "2.3": "d64d0954f8889f39",
 "2.4": "c7c51f8206acdad0",
 "2.5b": "280223b6a4ca8b9e",
 "2.6": "b5bea41b6c623f7c",
 "2.7": "d4c70e5e2e9e98c5",
 "2.8": "d22c98aac56cea2d",
 "2.9": "5d8b839d5a11f70f",
 "2.10": "b305560c5398c645",
 "2.11": "7c1b5e2043fb7485"
}
_RECORD = os.environ.get("LAB_RECORD")

def _norm(v):
    if isinstance(v, bool) or v is None: return v
    if isinstance(v, int): return v
    if isinstance(v, float): return round(v, 2)
    if isinstance(v, str): return v.strip()
    if isinstance(v, dict): return {str(k): _norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
    if isinstance(v, (list, tuple, set)):
        items = [_norm(x) for x in v]
        return sorted(items, key=json.dumps) if isinstance(v, set) else items
    if hasattr(v, "item"): return _norm(v.item())          # numpy numbers
    raise TypeError(f"check() needs plain Python values, got {type(v).__name__}")

def check(task, value):
    normal = _norm(value)
    if _RECORD:
        rec = json.load(open(_RECORD)) if os.path.exists(_RECORD) else {}
        rec[task] = normal; json.dump(rec, open(_RECORD, "w"))
    ok = hashlib.sha256(json.dumps(normal).encode()).hexdigest()[:16] == _EXPECTED.get(task)
    print(f"✅ {task} correct" if ok else f"❌ {task} not yet: check your code and the task text again")

def check_true(task, condition, hint=""):
    print(f"✅ {task} correct" if condition else f"❌ {task} not yet. {hint}")


Load the data the way Lab 01 taught you (given):

In [ ]:
trips = spark.read.parquet("data/yellow_tripdata_2024-01.parquet")
zones = (spark.read.option("header", True)
         .schema("LocationID int, Borough string, Zone string, service_zone string")
         .csv("data/taxi_zone_lookup.csv"))
trips.createOrReplaceTempView("trips")
zones.createOrReplaceTempView("zones")

## Warm-up (from Lab 01)

**2.1 How many trips have a negative *fare_amount*, per *VendorID*?** *neg_by_vendor* = list of `(VendorID, count)` sorted by VendorID. Guess the number of stages, then measure it into *neg_stages*.

In [ ]:
check("2.1", [neg_by_vendor, neg_stages])

## A. One question, two syntaxes

**2.2 With SQL: which 3 pickup boroughs have the highest average tip percentage on credit-card trips?**
* credit card: *payment_type* = 1; use only trips with *fare_amount* of at least 5
* tip percentage = 100 × *tip_amount* / *fare_amount*, rounded to 2 decimals
* use `spark.sql` over the views *trips* and *zones*; keep the DataFrame in *sql_df* and the rows `(Borough, avg_tip_pct)` in *sql_rows*

In [ ]:
check("2.2", sql_rows)

**2.3 Answer the same question with the DataFrame API (*api_df*). Are the physical plans of the two versions the same?** Compare them with `operators()`.

In [ ]:
check("2.3", [api_rows, same_plan])

## B. Read the plan

**2.4 Read the plan of query *q*:**
* which filters are pushed down to the trips scan? copy the *PushedFilters* into *pushed*
* which table does Spark broadcast, `"trips"` or `"zones"`? → *built*
* how many shuffle `Exchange` nodes are there (not `BroadcastExchange`)? → *n_shuffles*

In [ ]:
# given
q = (trips.join(zones, trips.PULocationID == zones.LocationID)
       .filter(F.col("fare_amount") > 50)
       .groupBy("Borough").count())



In [ ]:
check("2.4", [sorted(pushed), built, n_shuffles])

## C. Watch Catalyst decide

**2.5 Which Catalyst rules changed the plan of *wasteful*, and which rule removed `1 + 1 = 2`?** Use `rules_fired()` for *fired*; put the rule name in *folding_rule*.

In [ ]:
# given
wasteful = (trips.select("*")
              .join(zones, trips.PULocationID == zones.LocationID)
              .filter((F.col("fare_amount") > 50) & (F.lit(1) + F.lit(1) == 2))
              .select("Borough", "fare_amount"))



In [ ]:
check_true("2.5", {"PushDownPredicates", "ColumnPruning", "ConstantFolding"} <= fired, "run rules_fired on wasteful")
check("2.5b", folding_rule)

**2.6 Turn off the rule `PushDownPredicates` and build *wasteful* again. Is the *fare_amount* > 50 filter now above the join?** Answer True or False in *filter_above_join*, then turn the rule back on.

In [ ]:
check("2.6", filter_above_join)

## D. Pushdown, pruning and the UDF wall

**2.7 Filter trips to *fare_amount* over 100 twice: with a normal column condition and with a Python UDF. What is pushed down to the scan in each case, and which is faster?** Put the *PushedFilters* in *pushed_builtin* and *pushed_udf* (`[]` if none) and time both `count()`s with `timeit`.

In [ ]:
from pyspark.sql.functions import udf
over_100 = udf(lambda f: f is not None and f > 100, "boolean")   # given



In [ ]:
check("2.7", [sorted(pushed_builtin), pushed_udf])
check_true("2.7b", t_udf > t_builtin, "the UDF version should be slower")

**2.8 Which columns does Spark read from the file for `trips.agg(F.sum("fare_amount"), F.sum("tip_amount"))`?** Copy them from *ReadSchema* into *read_cols* (sorted names).

In [ ]:
check("2.8", read_cols)

## E. Tungsten: whole-stage codegen

**2.9 How many whole-stage codegen regions `*(n)` does the plan of *neg_q* have, and which operator separates them?** Put the answers in *n_codegen* and *separator*.

In [ ]:
check("2.9", [n_codegen, separator])

## F. DataFrame vs. RDD

**2.10 How many January 2024 trips started in each hour of the day (0–23)?** Count them twice, with the RDD API and with the DataFrame API, into dicts `{hour: count}` (*rdd_counts*, *df_counts*). Which one is faster?

In [ ]:
check("2.10", df_counts)
check_true("2.10b", rdd_counts == df_counts and t_rdd > t_df, "both dicts must match, and the RDD version should be slower")

## G. Statistics change the estimate

**2.11 How many trips have *fare_amount* over 100 on the table *trips_t*, and how close is the optimizer's guess?** Use `estimate()` three times:
* with no statistics → *est_none*
* after `ANALYZE TABLE ... FOR COLUMNS fare_amount` with `spark.sql.cbo.enabled` on → *est_uniform*
* after turning on `spark.sql.statistics.histogram.enabled` and analyzing again → *est_histogram*

Put the real count in *actual*. Guess first: will the plain statistics be close?

In [ ]:
# given: save trips as the table trips_t (rmtree avoids LOCATION_ALREADY_EXISTS on a rerun)
import shutil
shutil.rmtree("spark-warehouse/trips_t", ignore_errors=True)
trips.write.mode("overwrite").saveAsTable("trips_t")



In [ ]:
check("2.11", actual)
check_true("2.11b", est_none[1] is None and est_uniform[1] > 100 * actual and actual / 2 <= est_histogram[1] <= actual * 2,
           "no row count without stats; plain stats wildly off (they assume fares are spread evenly between min and max); a histogram within 2x")

---
Done? In the Theory part of Day 2, answer the red checklist cards aloud, using the plans you just read as evidence.